In [ ]:
!pip install -q ultralytics roboflow pandas matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 22.3 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
from roboflow import Roboflow

# Mount Google Drive to save models safely
drive.mount('/content/drive')

# Download the dataset
rf = Roboflow(api_key="eXwXDALxNZwhk3J0bxbT")
project = rf.workspace("drone-wxuiq").project("drone-vs-bird-combined")
dataset = project.version(1).download("yolov8")

Mounted at /content/drive
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to drone-vs-bird-combined-1 in yolov8:: 100%|██████████| 18214/18214 [00:07<00:00, 2396.81it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
from ultralytics import YOLO
import pandas as pd
import torch
import gc
import os

# Define the save path in Google Drive
drive_save_dir = '/content/drive/MyDrive/YOLO_Drone_Bird_Models_Phase1'
os.makedirs(drive_save_dir, exist_ok=True)

# List of architectures
architectures = [
    # 'yolov8n.pt',
    # 'yolov8s.pt',
    # 'yolov8n-p2',
    # 'yolo11n.pt',
    # 'yolo11s.pt',
    # 'yolov12n.pt',
    # 'yolo26n.pt',
    'yolo26s.pt'
]

epochs = 30
comparison_metrics = []

for model_name in architectures:
    print(f"\n{'='*50}")
    print(f"⚡ FAST TRACK: Phase 1 Training for: {model_name}")
    print(f"{'='*50}\n")

    # 1. Clean up GPU memory
    torch.cuda.empty_cache()
    gc.collect()

    # 2. Load the specific YOLO architecture
    if "-p2" in model_name:
        base_name = model_name.replace('-p2', '')
        model = YOLO(f"{model_name}.yaml").load(f"{base_name}.pt")
        clean_name = model_name
        size_category = "N (P2)"
    else:
        model = YOLO(model_name)
        clean_name = model_name.replace('.pt', '')
        size_category = model_name.split('.')[0][-1].upper()

    # 3. Dynamic batch sizing
    batch_size = 8 if 's.pt' in model_name else 16

    # 4. Train the model with SPEED OPTIMIZATIONS
    train_results = model.train(
        data=f"{dataset.location}/data.yaml",
        epochs=epochs,
        imgsz=640,
        batch=batch_size,
        project=drive_save_dir,
        name=f"{clean_name}_640px",

        # --- THE SPEED BOOSTERS ---
        cache=False,      # Loads entire dataset into RAM (Massive speed up)
        workers=8,       # Max out CPU data-loading threads
        amp=True,        # Force Automatic Mixed Precision (16-bit)
        plots=True      # Skip generating charts to save processing time
    )

    # 5. Extract validation metrics
    map50 = train_results.box.map50
    map50_95 = train_results.box.map
    inference_speed = train_results.speed['inference']

    # 6. Store results
    comparison_metrics.append({
        "Architecture": clean_name.upper(),
        "Size": size_category,
        "mAP@50": round(map50, 4),
        "mAP@50-95": round(map50_95, 4),
        "Inference Speed (ms)": round(inference_speed, 2)
    })

# --- Display the Final Comparison ---
df = pd.DataFrame(comparison_metrics)
df = df.sort_values(by=["mAP@50-95", "Inference Speed (ms)"], ascending=[False, True]).reset_index(drop=True)

print("\n\n📊 PHASE 1 PERFORMANCE COMPARISON (640px Resolution) 📊")
print(df.to_markdown(index=False))


⚡ FAST TRACK: Phase 1 Training for: yolo26s.pt

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drone-vs-bird-combined-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo26s_640px, nbs=64, nms=False, opset=None, optimi